In [6]:
import pandas as pd
import os
from Aplicacion_utils import *

In [2]:
# def extract_pixels_in_marmenor(folder_path, target_dates, grouping, net_set, polygon_path):
#     """
#     Extrae valores de píxeles dentro del área del Mar Menor a partir de GeoTIFFs y un polígono de máscara.

#     Args:
#         folder_path: carpeta con los TIFFs
#         target_dates: lista de fechas (formato YYYY-MM-DD)
#         grouping: tamaño de agrupamiento ("1x1", "3x3", "5x5", etc.)
#         net_set: prefijo del conjunto (C2X-Complex, C2X, C2RCC)
#         polygon_path: ruta al archivo GeoJSON o Shapefile del Mar Menor

#     Returns:
#         pd.DataFrame con bandas y coordenadas por píxel
#     """
#     results = []
#     target_dates = sorted(set(target_dates))

#     # Selección de ficheros TIFF
#     if net_set == "C2X-Complex":
#         tif_files = [f for f in os.listdir(folder_path) if f.endswith('.tif') and 'C2XComplexNets' in f]
#     elif net_set == "C2X":
#         tif_files = [f for f in os.listdir(folder_path) if f.endswith('.tif') and 'C2XNets' in f]
#     elif net_set == "C2RCC":
#         tif_files = [f for f in os.listdir(folder_path) if f.endswith('.tif') and 'C2RCC' in f]
#     else:
#         raise ValueError("Net Set no reconocido")

#     # Leer el polígono del Mar Menor
#     #gdf = gpd.read_file(polygon_path)
#     with open(polygon_path, "r") as f:
#         data = json.load(f)

#     gdf = gpd.GeoDataFrame.from_features(data["features"])
#     if gdf.crs is None:
#         gdf.set_crs("EPSG:32630", inplace=True)


#     for date in target_dates:
#         date_str = date.replace('-', '')
#         matching = [f for f in tif_files if date_str in f]
#         if not matching:
#             continue

#         tiff_file = os.path.join(folder_path, matching[0])
#         with rasterio.open(tiff_file) as dataset:
#             print(f"Procesando {tiff_file}")
#             bands = dataset.read()  # shape: (n_bands, height, width)

#             # Crear máscara booleana de los píxeles dentro del polígono
#             mask = features.geometry_mask(
#                 geometries=gdf.geometry,
#                 transform=dataset.transform,
#                 invert=True,
#                 out_shape=(dataset.height, dataset.width)
#             )

#             if grouping == "3x3":
#                 offset = 1
#             elif grouping == "5x5":
#                 offset = 2
#             elif grouping == "9x9":
#                 offset = 4
#             elif grouping == "15x15":
#                 offset = 7
#             else:
#                 offset = 0

#             # Iterar sobre todos los píxeles dentro del polígono
#             idxs = np.argwhere(mask)

#             for row_idx, col_idx in idxs:
#                 try:
#                     if offset > 0:
#                         row_start = max(row_idx - offset, 0)
#                         row_end = min(row_idx + offset + 1, dataset.height)
#                         col_start = max(col_idx - offset, 0)
#                         col_end = min(col_idx + offset + 1, dataset.width)

#                         window = (
#                             slice(row_start, row_end),
#                             slice(col_start, col_end)
#                         )

#                         reflectances = bands[:, window[0], window[1]]
#                         values = np.median(reflectances, axis=(1, 2))
#                     else:
#                         values = bands[:, row_idx, col_idx]

#                     # Convertir coordenadas a UTM (o lat/lon si prefieres)
#                     lon, lat = dataset.xy(row_idx, col_idx)

#                     results.append({
#                         "Date": date,
#                         "Latitude": lat,
#                         "Longitude": lon,
#                         **{f"Band_{i+1}": val for i, val in enumerate(values)}
#                     })
#                 except Exception as e:
#                     print(f"Error en píxel ({row_idx},{col_idx}): {e}")
#                     continue

#     return pd.DataFrame(results)


In [2]:
# a = extract_pixels_in_marmenor(
#     folder_path="Copernicus/SAFE_downloads/application",
#     target_dates=["2022-07-14"],
#     grouping="9x9",
#     net_set="C2X-Complex",
#     polygon_path="saved_files/marmenor_polygon.geojson"
# )

# Ojo que tarda unos 2 min en cargar el df - son ~1.2M de filas

Procesando Copernicus/SAFE_downloads/application/S2B_MSIL1C_20220714T104629_N0510_R051_T30SXG_20240723T192911_C2XComplexNets_10m.tif


KeyboardInterrupt: 

In [9]:
folder_path = "Copernicus/SAFE_downloads/application"
polygon_path="saved_files/marmenor_polygon.geojson"
# Hacer las fechas de una en una porque pesan mucho
target_dates = [
    '2016-04-01' #, '2022-07-14','2023-04-20'
]

groupings = ["5x5", "9x9"]
net_set = ["C2X-Complex"]


In [10]:

# Tarda unos dos minutos por cada dataframe

for grouping in groupings:
    for net in net_set:
        df_tiffs = extract_pixels_in_marmenor(folder_path, target_dates, grouping, net, polygon_path)
        df_tiffs["Date"] = pd.to_datetime(df_tiffs["Date"])
        df_tiffs.to_csv(f"saved_files/application/tmp_csv/df_tifs_{net}_{grouping}_{target_dates[0]}.csv", index=False)

Procesando Copernicus/SAFE_downloads/application/S2A_MSIL1C_20160401T105022_N0500_R051_T30SXG_20231030T102033_C2XComplexNets_10m.tif
Procesando Copernicus/SAFE_downloads/application/S2A_MSIL1C_20160401T105022_N0500_R051_T30SXG_20231030T102033_C2XComplexNets_10m.tif


In [11]:
# Cargamos los csv de los tifs
path = "saved_files/application/tmp_csv"
dfs_tifs = {}
for archivo in os.listdir(path):
    if archivo.startswith("df_tifs_") and archivo.endswith(f"{target_dates[0]}.csv") and "planet" not in archivo:
        nombre_sin_extension = os.path.splitext(archivo)[0]  # sin .csv
        ruta_completa = os.path.join(path, archivo)
        # Guardamos el nombre del archivo sin la fecha
        dfs_tifs[nombre_sin_extension[:-11]] = pd.read_csv(ruta_completa)

# Limpiamos nulos
for nombre_df, df in dfs_tifs.items():
    for band_set in ["rhow", "rhown","rtoa"]:
        dfs_tifs[nombre_df] = df.dropna()

In [12]:
# Dataframes con procesado C2X 5x5, C2X 9x9 y TOA 9x9
dfs_tifs_all = create_processed_dfs(dfs_tifs)

In [13]:
dfs = add_band_combinations(dfs_tifs_all)

In [14]:
for nombre_df, df in dfs.items():
    dfs[nombre_df] = compactar_prefijos_columnas(df)

In [15]:
dfs = add_season(dfs)

## Aplicar modelos

In [16]:
carpeta_modelos = "training_results/models/"

selection = {
    'C2X-Complex_rhow_9x9_depth_in_0_1': 'XGB',
    'C2X-Complex_rhow_9x9_depth_in_1_2': 'CAT',
    'TOA_9x9_depth_in_2_3': 'KNN',
    'C2X-Complex_rhow_5x5_depth_in_3_4': 'RF',
}


df_out = pd.DataFrame(dfs["df_tifs_C2X-Complex_rhow_9x9"].loc[:, ["Date", "Latitude", "Longitude"]])

for dataset, model_name in selection.items():
    if model_name in ["XGB", "CAT"]:
        df_in = dfs["df_tifs_C2X-Complex_rhow_9x9"]
        #df_out = pd.DataFrame(dfs["df_tifs_C2X-Complex_rhow_9x9"].loc[:, ["Date", "Latitude", "Longitude"]])
    elif model_name == "KNN":
        df_in = dfs["df_tifs_TOA_9x9"]
        #df_out = pd.DataFrame(dfs["df_tifs_TOA_9x9"].loc[:, ["Date", "Latitude", "Longitude"]])
    elif model_name == "RF":
        df_in = dfs["df_tifs_C2X-Complex_rhow_5x5"]
        #df_out = pd.DataFrame(dfs["df_tifs_C2X-Complex_rhow_5x5"].loc[:, ["Date", "Latitude", "Longitude"]])

    print(dataset, model_name)
    depth = dataset[-3:]
    df_out[f"Chl_pred_{depth}"] = predict_with_model(
                                    df=df_in,
                                    models_dir=carpeta_modelos,
                                    dataset_name=dataset,
                                    model_name=model_name,
                                    clip_min=0.2,
                                    strict=True
                                )

C2X-Complex_rhow_9x9_depth_in_0_1 XGB
Inference with XGB for C2X-Complex_rhow_9x9_depth_in_0_1
C2X-Complex_rhow_9x9_depth_in_1_2 CAT
Inference with CAT for C2X-Complex_rhow_9x9_depth_in_1_2
TOA_9x9_depth_in_2_3 KNN
Inference with KNN for TOA_9x9_depth_in_2_3
C2X-Complex_rhow_5x5_depth_in_3_4 RF
Inference with RF for C2X-Complex_rhow_5x5_depth_in_3_4


In [17]:
df_out.loc[:,["Date", "Latitude", "Longitude", "Chl_pred_0_1", "Chl_pred_1_2", "Chl_pred_2_3", "Chl_pred_3_4"]].to_csv(f"saved_files/application/preds/{target_dates[0]}_pred.csv", index = False)

In [18]:
df_out.describe()

,Date,Latitude,Longitude,Chl_pred_0_1,Chl_pred_1_2,Chl_pred_2_3,Chl_pred_3_4
count,1148384,1.148384e+06,1.148384e+06,1.148384e+06,1.148384e+06,1.148384e+06,1.148384e+06
mean,2016-04-01 00:00:00,4.177006e+06,6.949814e+05,2.632215e+00,4.172235e+00,1.212770e+00,5.081811e+00
min,2016-04-01 00:00:00,4.167645e+06,6.887050e+05,2.000000e-01,2.000000e-01,2.161788e-01,5.871628e-01
25%,2016-04-01 00:00:00,4.173405e+06,6.931950e+05,2.327621e+00,3.826546e+00,7.002827e-01,4.815651e+00
50%,2016-04-01 00:00:00,4.177035e+06,6.951450e+05,2.708559e+00,4.378032e+00,1.134678e+00,5.037987e+00
75%,2016-04-01 00:00:00,4.180215e+06,6.969150e+05,2.981557e+00,4.639265e+00,1.434943e+00,5.345276e+00
max,2016-04-01 00:00:00,4.187805e+06,7.006350e+05,1.251802e+01,1.392794e+01,2.088535e+01,1.111584e+01
std,NaN,4.524588e+03,2.501011e+03,5.259530e-01,7.439454e-01,8.015224e-01,6.235905e-01
